In [ ]:
# Replace only these four paths before running the notebook.
from pathlib import Path

BM25_WHEEL_PATH = Path(
    '/kaggle/input/<offline-packages-dataset>/bm25s-0.3.11-py3-none-any.whl'
)
LEGALIR_SOURCE_PATH = Path('/kaggle/input/<legalir-dataset>/train.json')
CORPUS_PATH = Path('/kaggle/input/<legalir-dataset>/selected-contexts')
MODEL_PATH = Path('/kaggle/input/<bge-reranker-dataset>')


In [ ]:
# Validate attached inputs, set offline mode, and install the pinned BM25 wheel.
import os
import subprocess
import sys
from pathlib import Path

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

if not BM25_WHEEL_PATH.is_file():
    raise FileNotFoundError(f'Attach the offline BM25 wheel at: {BM25_WHEEL_PATH}')
if not LEGALIR_SOURCE_PATH.is_file():
    raise FileNotFoundError(f'Attach the LegalIR source JSON at: {LEGALIR_SOURCE_PATH}')
if not CORPUS_PATH.exists():
    raise FileNotFoundError(f'Attach the LegalIR corpus at: {CORPUS_PATH}')
if not MODEL_PATH.is_dir():
    raise FileNotFoundError(f'Attach the complete model snapshot directory at: {MODEL_PATH}')

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
        str(BM25_WHEEL_PATH),
    ],
    check=True,
)


In [ ]:
# Standalone LegalIR holdout retrieval, reranking, and aggregate evaluation.
import json
import re
from collections import Counter
from collections.abc import Callable
from hashlib import sha256
from math import isfinite
from pathlib import Path
from statistics import median
from time import perf_counter

import bm25s
import numpy as np
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer


SPLIT_VERSION = "legalir_split_v1"
EXPECTED_SOURCE_SHA256 = (
    "c39cde9e74977e350f1456e7d487aafe67d2bcbaa4fa26fcabd557fe635635b7"
)
EXPECTED_SPLIT_COUNTS = {"train": 4_941, "dev": 1_036, "holdout": 1_023}
EXPECTED_DOCUMENTS = 8_532
EXPECTED_CHUNKS = 199_816

CHUNK_SIZE = 2_000
CHUNK_OVERLAP = 200
BM25_VERSION = "0.3.11"
BM25_METHOD = "lucene"
BM25_K1 = 1.5
BM25_B = 0.75
TOP_K_CHUNKS = 2_000
CANDIDATE_DEPTH = 100
SUPPORTING_CHUNKS_PER_DOCUMENT = 2

MODEL_NAME = "BAAI/bge-reranker-v2-m3"
DECLARED_MODEL_REVISION = (
    "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"
)
DEVICE = "cuda"
DTYPE_NAME = "float16"
BATCH_SIZE = 128
MAX_SEQUENCE_LENGTH = 8_192
SMOKE_QUERIES = 2

TOKEN_PATTERN = re.compile(r"\w+", flags=re.UNICODE)


def read_json(path: Path):
    if not path.is_file():
        raise FileNotFoundError(f"Required JSON file does not exist: {path}")
    try:
        with path.open(encoding="utf-8-sig") as stream:
            return json.load(stream)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON") from exc


def load_legalir(path: Path) -> dict:
    data = read_json(path)
    if not isinstance(data, dict) or not all(
        isinstance(sample, dict) for sample in data.values()
    ):
        raise ValueError("LegalIR source must be an object keyed by sample ID")
    return data


def corpus_documents(data, source: str) -> list[dict]:
    if isinstance(data, dict):
        documents = [data]
    elif isinstance(data, list):
        documents = data
    else:
        raise ValueError(f"{source}: expected a document or list of documents")
    if not all(isinstance(document, dict) for document in documents):
        raise ValueError(f"{source}: every corpus document must be an object")
    return documents


def load_corpus(path: Path) -> list[dict]:
    if path.is_dir():
        json_paths = sorted(
            candidate
            for candidate in path.rglob("*")
            if candidate.is_file() and candidate.suffix.lower() == ".json"
        )
        if not json_paths:
            raise ValueError("Corpus directory contains no JSON files")
        documents = []
        for json_path in json_paths:
            documents.extend(corpus_documents(read_json(json_path), str(json_path)))
        return documents
    if path.is_file() and path.suffix.lower() == ".json":
        return corpus_documents(read_json(path), str(path))
    raise ValueError("Corpus input must be a JSON file or a directory of JSON files")


def make_legalir_split(samples: dict) -> dict[str, list[str]]:
    """Hash exact raw questions; do not normalize question text."""

    split_ids = {"train": [], "dev": [], "holdout": []}
    for sample_id, sample in samples.items():
        canonical_id = str(sample_id)
        question = sample.get("question")
        group_key = (
            question
            if isinstance(question, str)
            else f"\0fallback-sample-id:{canonical_id}"
        )
        bucket = int(sha256(group_key.encode("utf-8")).hexdigest()[:8], 16) % 100
        split_name = (
            "train" if bucket < 70 else "dev" if bucket < 85 else "holdout"
        )
        split_ids[split_name].append(canonical_id)

    for ids in split_ids.values():
        ids.sort()
    return split_ids


def prepare_holdout_samples(source_path: Path) -> tuple[dict, dict]:
    source_hash = sha256(source_path.read_bytes()).hexdigest()
    if source_hash != EXPECTED_SOURCE_SHA256:
        raise RuntimeError(
            "LegalIR source SHA-256 mismatch; stop without running the experiment"
        )

    samples = load_legalir(source_path)
    invalid_questions = sum(
        not isinstance(sample.get("question"), str) for sample in samples.values()
    )
    invalid_labels = sum(
        not isinstance(sample.get("answer"), list) or not sample.get("answer")
        for sample in samples.values()
    )
    if invalid_questions or invalid_labels:
        raise ValueError(
            "Expected labeled LegalIR data with string questions and non-empty "
            f"answer lists; invalid question count={invalid_questions}, "
            f"invalid answer count={invalid_labels}"
        )

    split_ids = make_legalir_split(samples)
    counts = {name: len(ids) for name, ids in split_ids.items()}
    if counts != EXPECTED_SPLIT_COUNTS:
        raise RuntimeError(
            f"Split counts mismatch: expected {EXPECTED_SPLIT_COUNTS}, got {counts}; "
            "stop without running the experiment"
        )

    split_sets = {name: set(ids) for name, ids in split_ids.items()}
    if (
        split_sets["train"] & split_sets["dev"]
        or split_sets["train"] & split_sets["holdout"]
        or split_sets["dev"] & split_sets["holdout"]
        or len(set().union(*split_sets.values())) != len(samples)
    ):
        raise RuntimeError("Split ID disjointness/completeness validation failed")

    canonical_samples = {
        str(sample_id): sample for sample_id, sample in samples.items()
    }
    holdout_samples = {
        sample_id: canonical_samples[sample_id]
        for sample_id in split_ids["holdout"]
    }
    if len(holdout_samples) != EXPECTED_SPLIT_COUNTS["holdout"]:
        raise RuntimeError("Fixed holdout selection failed")

    return holdout_samples, {
        "version": SPLIT_VERSION,
        "source_sha256": source_hash,
        "holdout_queries": len(holdout_samples),
    }


def validate_window_parameters(chunk_size: int, overlap: int) -> None:
    if not isinstance(chunk_size, int) or isinstance(chunk_size, bool):
        raise TypeError("chunk_size must be an integer")
    if not isinstance(overlap, int) or isinstance(overlap, bool):
        raise TypeError("overlap must be an integer")
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Require chunk_size > overlap >= 0")


def chunk_document(
    document: dict,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
) -> list[dict]:
    validate_window_parameters(chunk_size, overlap)
    document_id = str(document["id"])
    text = document["passage"]
    if not isinstance(text, str):
        raise TypeError("Every corpus passage must be a string")
    if not text:
        return []

    step = chunk_size - overlap
    chunks = []
    for chunk_index, char_start in enumerate(range(0, len(text), step)):
        char_end = min(char_start + chunk_size, len(text))
        chunks.append(
            {
                "chunk_id": f"{document_id}:{chunk_index}",
                "document_id": document_id,
                "text": text[char_start:char_end],
            }
        )
        if char_end == len(text):
            break
    return chunks


def chunk_corpus(
    documents: list[dict],
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
) -> list[dict]:
    validate_window_parameters(chunk_size, overlap)
    chunks = []
    seen_document_ids = set()
    for document in documents:
        document_id = str(document["id"])
        if document_id in seen_document_ids:
            raise ValueError("Corpus document IDs must be unique")
        seen_document_ids.add(document_id)
        chunks.extend(
            chunk_document(document, chunk_size=chunk_size, overlap=overlap)
        )
    return chunks


def lexical_tokenize(text: str) -> list[str]:
    if not isinstance(text, str):
        raise TypeError("Query text must be a string")
    return TOKEN_PATTERN.findall(text.lower())


def sparse_index_size_bytes(retriever: bm25s.BM25) -> int:
    return sum(
        value.nbytes
        for value in retriever.scores.values()
        if isinstance(value, np.ndarray)
    )


def build_bm25(chunks: list[dict]) -> dict:
    if bm25s.__version__ != BM25_VERSION:
        raise RuntimeError(
            f"Expected bm25s=={BM25_VERSION}, got bm25s=={bm25s.__version__}"
        )
    if not chunks:
        raise ValueError("Chunks must not be empty")

    tokenized_chunks = bm25s.tokenize(
        [chunk["text"] for chunk in chunks],
        lower=True,
        token_pattern=r"(?u)\w+",
        stopwords=[],
        stemmer=None,
        return_ids=True,
        show_progress=False,
    )
    retriever = bm25s.BM25(k1=BM25_K1, b=BM25_B, method=BM25_METHOD)
    retriever.index(tokenized_chunks, show_progress=False)

    return {
        "retriever": retriever,
        "chunk_metadata": [
            (str(chunk["chunk_id"]), str(chunk["document_id"]))
            for chunk in chunks
        ],
        "number_of_chunks": len(chunks),
        "sparse_index_size_bytes": sparse_index_size_bytes(retriever),
    }


def aggregate_documents_sum_top_2(chunk_hits: list[dict]) -> list[dict]:
    grouped = {}
    for hit in chunk_hits:
        document_id = str(hit["document_id"])
        score = float(hit["score"])
        chunk_rank = int(hit["chunk_rank"])
        if not isfinite(score) or chunk_rank <= 0:
            raise ValueError("BM25 hit score/rank is invalid")
        document = grouped.setdefault(
            document_id,
            {"scores": [], "best_chunk_rank": chunk_rank},
        )
        document["scores"].append(score)
        document["best_chunk_rank"] = min(
            document["best_chunk_rank"], chunk_rank
        )

    ranked_documents = []
    for document_id, document in grouped.items():
        scores = sorted(document["scores"], reverse=True)
        ranked_documents.append(
            {
                "document_id": document_id,
                "score": sum(scores[:2]),
                "best_chunk_rank": document["best_chunk_rank"],
            }
        )

    ranked_documents.sort(
        key=lambda document: (
            -document["score"],
            document["best_chunk_rank"],
            document["document_id"],
        )
    )
    return ranked_documents


def build_fixed_candidates(
    index: dict,
    chunks: list[dict],
    samples: dict,
    retrieval_batch_size: int = 64,
) -> dict:
    if TOP_K_CHUNKS != 2_000 or CANDIDATE_DEPTH != 100:
        raise RuntimeError("Frozen retrieval depth constants changed")
    if len(chunks) != index["number_of_chunks"]:
        raise ValueError("Chunks must exactly match the BM25 index")
    if TOP_K_CHUNKS > index["number_of_chunks"]:
        raise ValueError("top_k_chunks exceeds the indexed chunk count")

    sample_items = list(samples.items())
    candidates_by_query = {}
    retrieval_started = perf_counter()

    for batch_start in range(0, len(sample_items), retrieval_batch_size):
        batch = sample_items[batch_start : batch_start + retrieval_batch_size]
        query_tokens = [
            lexical_tokenize(sample["question"]) for _, sample in batch
        ]
        retrieval_result = index["retriever"].retrieve(
            query_tokens,
            k=TOP_K_CHUNKS,
            sorted=True,
            return_as="tuple",
            show_progress=False,
        )

        for (sample_id, _), hit_indices, hit_scores in zip(
            batch, retrieval_result.documents, retrieval_result.scores
        ):
            chunk_hits = []
            for chunk_rank, (chunk_index_value, score_value) in enumerate(
                zip(hit_indices, hit_scores), start=1
            ):
                chunk_index = int(chunk_index_value)
                chunk_id, document_id = index["chunk_metadata"][chunk_index]
                chunk = chunks[chunk_index]
                if (
                    str(chunk["chunk_id"]) != chunk_id
                    or str(chunk["document_id"]) != document_id
                ):
                    raise RuntimeError("Chunk order does not match the BM25 index")
                chunk_hits.append(
                    {
                        "chunk_id": chunk_id,
                        "document_id": document_id,
                        "score": float(score_value),
                        "chunk_rank": chunk_rank,
                        "text": chunk["text"],
                    }
                )

            aggregated = aggregate_documents_sum_top_2(chunk_hits)
            selected_documents = aggregated[:CANDIDATE_DEPTH]
            if len(selected_documents) != CANDIDATE_DEPTH:
                raise RuntimeError("A query did not produce exactly 100 documents")

            by_document = {
                document["document_id"]: [] for document in selected_documents
            }
            for hit in chunk_hits:
                supporting_chunks = by_document.get(hit["document_id"])
                if (
                    supporting_chunks is not None
                    and len(supporting_chunks) < SUPPORTING_CHUNKS_PER_DOCUMENT
                ):
                    supporting_chunks.append(
                        {
                            "text": hit["text"],
                            "bm25_rank": hit["chunk_rank"],
                            "bm25_score": hit["score"],
                        }
                    )

            query_candidates = []
            for original_rank, document in enumerate(
                selected_documents, start=1
            ):
                supporting_chunks = by_document[document["document_id"]]
                if not 1 <= len(supporting_chunks) <= 2:
                    raise RuntimeError(
                        "Every candidate must have one or two supporting chunks"
                    )
                query_candidates.append(
                    {
                        "document_id": document["document_id"],
                        "original_rank": original_rank,
                        "supporting_chunks": supporting_chunks,
                    }
                )

            candidate_ids = [
                candidate["document_id"] for candidate in query_candidates
            ]
            if len(candidate_ids) != len(set(candidate_ids)):
                raise RuntimeError("Candidate document IDs must be unique")
            candidates_by_query[str(sample_id)] = query_candidates

    return {
        "candidates_by_query": candidates_by_query,
        "reference_rankings": {
            sample_id: [
                candidate["document_id"] for candidate in candidates
            ]
            for sample_id, candidates in candidates_by_query.items()
        },
        "retrieval_seconds": perf_counter() - retrieval_started,
    }


def load_reranker(model_path: Path) -> dict:
    if DEVICE != "cuda" or DTYPE_NAME != "float16":
        raise RuntimeError("Frozen device/dtype constants changed")
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required by the frozen holdout configuration")

    resolved_model_path = model_path.expanduser().resolve()
    if not resolved_model_path.is_dir():
        raise FileNotFoundError("Complete local model snapshot is unavailable")

    config_path = resolved_model_path / "config.json"
    raw_config = read_json(config_path)
    config_commit_hash = raw_config.get("_commit_hash")
    if config_commit_hash is None:
        revision_status = "declared-offline-snapshot"
    elif (
        isinstance(config_commit_hash, str)
        and config_commit_hash == DECLARED_MODEL_REVISION
    ):
        revision_status = "verified-from-config"
    else:
        raise RuntimeError(
            "Local model config _commit_hash does not match the declared revision"
        )

    model_load_started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(
        str(resolved_model_path),
        local_files_only=True,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        str(resolved_model_path),
        dtype=torch.float16,
        local_files_only=True,
    )
    model.to(DEVICE)
    model.eval()

    tokenizer_limit = int(tokenizer.model_max_length)
    model_limit = int(
        getattr(model.config, "max_position_embeddings", MAX_SEQUENCE_LENGTH)
    )
    if tokenizer_limit < MAX_SEQUENCE_LENGTH or model_limit < MAX_SEQUENCE_LENGTH:
        raise RuntimeError(
            "Local model/tokenizer does not support the frozen length of 8192"
        )

    return {
        "model": model,
        "tokenizer": tokenizer,
        "model_name": MODEL_NAME,
        "declared_revision": DECLARED_MODEL_REVISION,
        "config_commit_hash": config_commit_hash,
        "revision_status": revision_status,
        "model_input_path": str(resolved_model_path),
        "device": DEVICE,
        "dtype": DTYPE_NAME,
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "load_seconds": perf_counter() - model_load_started,
    }


def score_query_chunk_pairs(
    reranker: dict,
    pairs: list[tuple[str, str]],
    batch_size: int,
    progress: Callable[[int, int], None] | None = None,
) -> dict:
    if batch_size != BATCH_SIZE:
        raise RuntimeError("Cross-encoder batch size must remain 128")
    if not pairs:
        raise ValueError("At least one query/chunk pair is required")

    tokenizer = reranker["tokenizer"]
    model = reranker["model"]
    scores = []
    token_lengths = []
    model_forward_seconds = 0.0
    scoring_started = perf_counter()

    for batch_start in range(0, len(pairs), batch_size):
        batch = pairs[batch_start : batch_start + batch_size]
        questions = [question for question, _ in batch]
        passages = [passage for _, passage in batch]

        untruncated = tokenizer(
            questions,
            passages,
            padding=False,
            truncation=False,
            return_length=True,
        )
        token_lengths.extend(int(length) for length in untruncated["length"])

        inputs = tokenizer(
            questions,
            passages,
            padding=True,
            truncation="only_second",
            max_length=MAX_SEQUENCE_LENGTH,
            return_tensors="pt",
        )
        inputs = {name: value.to(DEVICE) for name, value in inputs.items()}

        torch.cuda.synchronize()
        forward_started = perf_counter()
        with torch.inference_mode():
            logits = model(**inputs, return_dict=True).logits.view(-1).float()
        torch.cuda.synchronize()
        model_forward_seconds += perf_counter() - forward_started

        batch_scores = logits.cpu().tolist()
        if len(batch_scores) != len(batch) or not all(
            isfinite(score) for score in batch_scores
        ):
            raise RuntimeError("Cross-encoder returned invalid scores")
        scores.extend(float(score) for score in batch_scores)

        if progress is not None:
            progress(min(batch_start + len(batch), len(pairs)), len(pairs))

    lengths = np.asarray(token_lengths, dtype=np.int32)
    truncated_pairs = int(np.sum(lengths > MAX_SEQUENCE_LENGTH))
    scoring_seconds = perf_counter() - scoring_started
    return {
        "scores": scores,
        "diagnostics": {
            "number_of_pairs": len(pairs),
            "token_length_before_truncation": {
                "median": float(np.median(lengths)),
                "p95": float(np.percentile(lengths, 95)),
                "max": int(lengths.max()),
            },
            "truncated_pairs": truncated_pairs,
            "truncated_fraction": truncated_pairs / len(pairs),
            "model_forward_seconds": model_forward_seconds,
            "scoring_seconds": scoring_seconds,
            "model_forward_pairs_per_second": (
                len(pairs) / model_forward_seconds
            ),
            "end_to_end_pairs_per_second": len(pairs) / scoring_seconds,
        },
    }


def rerank_documents(
    candidates: list[dict],
    supporting_chunk_scores: list[list[float]],
) -> list[dict]:
    if len(candidates) != len(supporting_chunk_scores):
        raise ValueError("Each candidate must have one supporting-score list")

    candidate_ids = [
        str(candidate["document_id"]) for candidate in candidates
    ]
    if len(candidate_ids) != len(set(candidate_ids)):
        raise ValueError("Candidate document IDs must be unique")

    reranked = []
    for candidate, scores in zip(candidates, supporting_chunk_scores):
        expected_scores = len(candidate["supporting_chunks"])
        if not 1 <= expected_scores <= SUPPORTING_CHUNKS_PER_DOCUMENT:
            raise ValueError("Candidate must have one or two supporting chunks")
        if len(scores) != expected_scores:
            raise ValueError("Supporting-score count does not match chunks")
        canonical_scores = [float(score) for score in scores]
        if not all(isfinite(score) for score in canonical_scores):
            raise ValueError("Cross-encoder scores must be finite")
        reranked.append(
            {
                "document_id": candidate["document_id"],
                "original_rank": candidate["original_rank"],
                "cross_encoder_score": sum(canonical_scores),
                "cross_encoder_chunk_scores": canonical_scores,
            }
        )

    reranked.sort(
        key=lambda document: (
            -document["cross_encoder_score"],
            document["original_rank"],
            str(document["document_id"]),
        )
    )
    if {document["document_id"] for document in reranked} != set(candidate_ids):
        raise RuntimeError("Reranking changed the fixed candidate set")
    return reranked


def rerank_fixed_candidates(
    reranker: dict,
    samples: dict,
    candidates_by_query: dict[str, list[dict]],
    batch_size: int,
    progress: Callable[[int, int], None] | None = None,
) -> dict:
    pairs = []
    score_counts = []
    for sample_id, sample in samples.items():
        candidates = candidates_by_query.get(str(sample_id))
        if candidates is None:
            raise KeyError("A fixed candidate set is missing")
        query_counts = []
        for candidate in candidates:
            support_count = len(candidate["supporting_chunks"])
            query_counts.append(support_count)
            pairs.extend(
                (sample["question"], supporting_chunk["text"])
                for supporting_chunk in candidate["supporting_chunks"]
            )
        score_counts.append((str(sample_id), query_counts))

    scoring = score_query_chunk_pairs(
        reranker,
        pairs,
        batch_size=batch_size,
        progress=progress,
    )

    score_offset = 0
    reranked_by_query = {}
    for sample_id, query_counts in score_counts:
        document_scores = []
        for support_count in query_counts:
            document_scores.append(
                scoring["scores"][
                    score_offset : score_offset + support_count
                ]
            )
            score_offset += support_count
        reranked_by_query[sample_id] = rerank_documents(
            candidates_by_query[sample_id], document_scores
        )
    if score_offset != len(scoring["scores"]):
        raise RuntimeError("Not every cross-encoder score was consumed")

    return {
        "reranked_by_query": reranked_by_query,
        "rankings": {
            sample_id: [
                document["document_id"] for document in documents
            ]
            for sample_id, documents in reranked_by_query.items()
        },
        "diagnostics": scoring["diagnostics"],
    }


def make_legalir_predictions(
    rankings: dict[str, list[str]],
) -> dict[str, dict[str, list[str]]]:
    """Strict top-5 adapter for the bundled LegalIR scorer contract."""

    predictions = {}
    for sample_id, ranked_ids in rankings.items():
        top_5 = [str(document_id) for document_id in ranked_ids[:5]]
        assert 1 <= len(top_5) <= 5
        assert len(top_5) == len(set(top_5))
        predictions[sample_id] = {"answer": top_5}
    return predictions


def bundled_scorer_compatible_eval(
    predictions: dict,
    truth: dict,
) -> dict:
    """Inline the authoritative scoring/LegalIR/scoring.py semantics."""

    y_pred = {key: value["answer"] for key, value in predictions.items()}
    y_true = {key: value for key, value in truth.items()}

    ids_preds = []
    ids_truth = []
    for key in y_pred:
        ids_preds.append(key)
    for key in y_true:
        ids_truth.append(key)

    if len(ids_preds) != len(ids_truth):
        raise RuntimeError("Samples in predictions do not match the reference")

    recall = np.array(
        [
            len(set(y_true[key]) & set(y_pred.get(key, set()))) / len(y_true[key])
            if 0 < len(y_pred.get(key)) <= 5
            else 0
            for key in ids_truth
        ]
    ).mean()
    precision = np.array(
        [
            len(set(y_true[key]) & set(y_pred.get(key, set())))
            / len(y_pred[key])
            if 0 < len(y_pred.get(key)) <= 5
            else 0
            for key in ids_preds
        ]
    ).mean()
    return {"precision": float(precision), "recall": float(recall)}


def percentile(values: list[int], percent: int) -> float | None:
    if not values:
        return None
    ordered = sorted(values)
    position = (len(ordered) - 1) * percent / 100
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    fraction = position - lower
    return float(
        ordered[lower] + (ordered[upper] - ordered[lower]) * fraction
    )


def evaluate_internal(samples: dict, rankings: dict[str, list[str]]) -> dict:
    depths = (5, 10, 20, 50, 100)
    recall_values = {depth: [] for depth in depths}
    reciprocal_ranks = []

    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        ranked = [
            str(document_id)
            for document_id in rankings.get(str(sample_id), [])
        ]
        if len(ranked) != CANDIDATE_DEPTH:
            raise ValueError("Every evaluation ranking must contain 100 documents")
        if len(ranked) != len(set(ranked)):
            raise ValueError("Evaluation rankings must contain unique documents")

        for depth in depths:
            recall_values[depth].append(
                len(gold.intersection(ranked[:depth])) / len(gold)
            )

        first_gold_rank = next(
            (
                rank
                for rank, document_id in enumerate(ranked, start=1)
                if document_id in gold
            ),
            None,
        )
        reciprocal_ranks.append(
            0.0 if first_gold_rank is None else 1 / first_gold_rank
        )

    number_of_queries = len(samples)
    if number_of_queries != EXPECTED_SPLIT_COUNTS["holdout"]:
        raise RuntimeError("Internal evaluation requires the complete fixed holdout")

    return {
        "recall_at_5": sum(recall_values[5]) / number_of_queries,
        "recall_at_10": sum(recall_values[10]) / number_of_queries,
        "recall_at_20": sum(recall_values[20]) / number_of_queries,
        "recall_at_50": sum(recall_values[50]) / number_of_queries,
        "recall_at_100": sum(recall_values[100]) / number_of_queries,
        "mrr": sum(reciprocal_ranks) / number_of_queries,
        "mrr_scope": "fixed top-100; gold absent from top-100 gives reciprocal rank 0",
    }


def summarize_candidate_ceiling(
    samples: dict,
    rankings: dict[str, list[str]],
) -> dict:
    recalls = []
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        ranked = rankings[str(sample_id)]
        recalls.append(len(gold.intersection(ranked[:100])) / len(gold))
    return {
        "recall_at_100_mean": sum(recalls) / len(recalls),
        "zero_recall_rate_at_100": sum(value == 0 for value in recalls)
        / len(recalls),
        "full_recall_rate_at_100": sum(value == 1 for value in recalls)
        / len(recalls),
    }


def summarize_first_gold_ranks(
    samples: dict,
    rankings: dict[str, list[str]],
) -> dict:
    found_ranks = []
    bins = Counter()
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        ranked = rankings[str(sample_id)]
        first_gold_rank = next(
            (
                rank
                for rank, document_id in enumerate(ranked, start=1)
                if document_id in gold
            ),
            None,
        )
        if first_gold_rank is None:
            bins["not_found"] += 1
        else:
            found_ranks.append(first_gold_rank)
            if first_gold_rank == 1:
                bins["rank_1"] += 1
            elif first_gold_rank <= 5:
                bins["rank_2_5"] += 1
            elif first_gold_rank <= 10:
                bins["rank_6_10"] += 1
            elif first_gold_rank <= 20:
                bins["rank_11_20"] += 1
            elif first_gold_rank <= 50:
                bins["rank_21_50"] += 1
            else:
                bins["rank_51_100"] += 1

    return {
        "when_found": {
            "median": float(median(found_ranks)) if found_ranks else None,
            "p90": percentile(found_ranks, 90),
            "p95": percentile(found_ranks, 95),
        },
        "counts": {
            label: bins[label]
            for label in (
                "rank_1",
                "rank_2_5",
                "rank_6_10",
                "rank_11_20",
                "rank_21_50",
                "rank_51_100",
                "not_found",
            )
        },
    }


def directional_conclusion(
    precision_delta: float,
    recall_delta: float,
) -> str:
    if precision_delta > 0 and recall_delta > 0:
        return (
            "The DEV-selected zero-shot cross-encoder reranking configuration "
            "generalizes directionally on the fixed local holdout."
        )
    if (
        precision_delta > 0 > recall_delta
        or recall_delta > 0 > precision_delta
    ):
        return (
            "The fixed local holdout shows a precision/recall trade-off for "
            "the DEV-selected reranking configuration."
        )
    return (
        "The DEV reranking gain does not generalize directionally on the "
        "fixed local holdout."
    )


def run_holdout_experiment(
    legalir_source_path: Path,
    corpus_path: Path,
    model_path: Path,
) -> dict:
    if (
        CHUNK_SIZE != 2_000
        or CHUNK_OVERLAP != 200
        or BM25_K1 != 1.5
        or BM25_B != 0.75
        or TOP_K_CHUNKS != 2_000
        or CANDIDATE_DEPTH != 100
        or SUPPORTING_CHUNKS_PER_DOCUMENT != 2
        or BATCH_SIZE != 128
        or MAX_SEQUENCE_LENGTH != 8_192
    ):
        raise RuntimeError("A frozen holdout control has changed")

    holdout_samples, split_info = prepare_holdout_samples(
        Path(legalir_source_path)
    )
    documents = load_corpus(Path(corpus_path))
    if len(documents) != EXPECTED_DOCUMENTS:
        raise RuntimeError(
            f"Expected {EXPECTED_DOCUMENTS} corpus documents, got {len(documents)}"
        )

    chunks = chunk_corpus(
        documents,
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP,
    )
    if len(chunks) != EXPECTED_CHUNKS:
        raise RuntimeError(
            f"Expected {EXPECTED_CHUNKS} fixed chunks, got {len(chunks)}"
        )

    index = build_bm25(chunks)
    reranker = load_reranker(Path(model_path))

    # Smoke test: no sample-level identifiers, text, predictions, or metrics escape.
    smoke_samples = dict(list(holdout_samples.items())[:SMOKE_QUERIES])
    smoke_candidates = build_fixed_candidates(index, chunks, smoke_samples)
    smoke_reranked = rerank_fixed_candidates(
        reranker,
        smoke_samples,
        smoke_candidates["candidates_by_query"],
        batch_size=BATCH_SIZE,
    )
    smoke_candidate_sets_preserved = all(
        set(smoke_candidates["reference_rankings"][sample_id])
        == set(smoke_reranked["rankings"][sample_id])
        for sample_id in smoke_samples
    )
    smoke_scores_finite = all(
        isfinite(document["cross_encoder_score"])
        for documents_for_query in smoke_reranked["reranked_by_query"].values()
        for document in documents_for_query
    )
    smoke_deterministic_ordering = True
    for sample_id in smoke_samples:
        scored_by_id = {
            document["document_id"]: document
            for document in smoke_reranked["reranked_by_query"][sample_id]
        }
        original_candidates = smoke_candidates["candidates_by_query"][sample_id]
        score_lists = [
            scored_by_id[candidate["document_id"]][
                "cross_encoder_chunk_scores"
            ]
            for candidate in original_candidates
        ]
        repeated = rerank_documents(original_candidates, score_lists)
        smoke_deterministic_ordering &= [
            document["document_id"] for document in repeated
        ] == smoke_reranked["rankings"][sample_id]

    smoke_test = {
        "queries": SMOKE_QUERIES,
        "score_shape": [
            smoke_reranked["diagnostics"]["number_of_pairs"]
        ],
        "finite_scores": bool(smoke_scores_finite),
        "candidate_sets_preserved": bool(smoke_candidate_sets_preserved),
        "deterministic_ordering": bool(smoke_deterministic_ordering),
    }
    if not all(
        (
            smoke_test["finite_scores"],
            smoke_test["candidate_sets_preserved"],
            smoke_test["deterministic_ordering"],
        )
    ):
        raise RuntimeError("Aggregate smoke-test invariant failed; stop")

    fixed_candidates = build_fixed_candidates(
        index, chunks, holdout_samples
    )
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    last_progress_time = [perf_counter()]

    def show_progress(completed: int, total: int) -> None:
        now = perf_counter()
        if completed == total or now - last_progress_time[0] >= 30:
            print(f"reranking pairs: {completed}/{total}", flush=True)
            last_progress_time[0] = now

    reranked = rerank_fixed_candidates(
        reranker,
        holdout_samples,
        fixed_candidates["candidates_by_query"],
        batch_size=BATCH_SIZE,
        progress=show_progress,
    )
    reference_rankings = fixed_candidates["reference_rankings"]
    reranked_rankings = reranked["rankings"]

    identical_count = sum(
        set(reference_rankings[sample_id])
        == set(reranked_rankings[sample_id])
        for sample_id in holdout_samples
    )
    if (
        identical_count != EXPECTED_SPLIT_COUNTS["holdout"]
        or identical_count != len(holdout_samples)
    ):
        raise RuntimeError(
            "Candidate sets are not identical for 1023/1023 queries; "
            "stop without interpreting metrics"
        )

    reference_internal = evaluate_internal(
        holdout_samples, reference_rankings
    )
    reranked_internal = evaluate_internal(
        holdout_samples, reranked_rankings
    )
    reference_ceiling = summarize_candidate_ceiling(
        holdout_samples, reference_rankings
    )
    reranked_ceiling = summarize_candidate_ceiling(
        holdout_samples, reranked_rankings
    )
    if (
        reference_internal["recall_at_100"]
        != reranked_internal["recall_at_100"]
        or reference_ceiling != reranked_ceiling
    ):
        raise RuntimeError(
            "Reference and reranked Recall@100 ceilings differ; "
            "implementation bug, stop without interpreting metrics"
        )

    truth = {
        sample_id: sample["answer"]
        for sample_id, sample in holdout_samples.items()
    }
    reference_predictions = make_legalir_predictions(reference_rankings)
    reranked_predictions = make_legalir_predictions(reranked_rankings)
    reference_official = bundled_scorer_compatible_eval(
        reference_predictions, truth
    )
    reranked_official = bundled_scorer_compatible_eval(
        reranked_predictions, truth
    )

    deltas = {
        "precision": (
            reranked_official["precision"]
            - reference_official["precision"]
        ),
        "recall": (
            reranked_official["recall"] - reference_official["recall"]
        ),
        "mrr": reranked_internal["mrr"] - reference_internal["mrr"],
        "recall_at_10": (
            reranked_internal["recall_at_10"]
            - reference_internal["recall_at_10"]
        ),
        "recall_at_20": (
            reranked_internal["recall_at_20"]
            - reference_internal["recall_at_20"]
        ),
        "recall_at_50": (
            reranked_internal["recall_at_50"]
            - reference_internal["recall_at_50"]
        ),
        "recall_at_100": (
            reranked_internal["recall_at_100"]
            - reference_internal["recall_at_100"]
        ),
    }
    if deltas["recall_at_100"] != 0:
        raise RuntimeError("Recall@100 delta must be exactly zero")

    runtime = {
        "model_load_seconds": reranker["load_seconds"],
        "candidate_retrieval_seconds": fixed_candidates[
            "retrieval_seconds"
        ],
        **reranked["diagnostics"],
        "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated()),
    }

    # Only aggregate objects below are returned and serialized.
    return {
        "split": split_info,
        "controls": {
            "documents": len(documents),
            "chunks": len(chunks),
            "chunk_size": CHUNK_SIZE,
            "overlap": CHUNK_OVERLAP,
            "bm25_library": f"bm25s=={bm25s.__version__}",
            "bm25_method": BM25_METHOD,
            "k1": BM25_K1,
            "b": BM25_B,
            "top_k_chunks": TOP_K_CHUNKS,
            "aggregation": "sum top-2",
            "candidate_depth": CANDIDATE_DEPTH,
            "supporting_chunks_per_document": "up to 2",
        },
        "reranker": {
            "model": reranker["model_name"],
            "declared_revision": reranker["declared_revision"],
            "config_commit_hash": reranker["config_commit_hash"],
            "revision_status": reranker["revision_status"],
            "model_input_path": reranker["model_input_path"],
            "max_sequence_length": reranker["max_sequence_length"],
            "torch_version": torch.__version__,
            "transformers_version": transformers.__version__,
            "device": reranker["device"],
            "dtype": reranker["dtype"],
            "batch_size": BATCH_SIZE,
        },
        "smoke_test": smoke_test,
        "candidate_sets_identical": {
            "queries": identical_count,
            "total_queries": len(holdout_samples),
        },
        "candidate_recall_at_100_ceiling": {
            **reference_ceiling,
            "reference_reranked_identical": True,
            "recall_at_100_delta": deltas["recall_at_100"],
        },
        "reference": {
            "bundled_scorer": reference_official,
            "internal": reference_internal,
            "first_gold_rank": summarize_first_gold_ranks(
                holdout_samples, reference_rankings
            ),
        },
        "reranked": {
            "bundled_scorer": reranked_official,
            "internal": reranked_internal,
            "first_gold_rank": summarize_first_gold_ranks(
                holdout_samples, reranked_rankings
            ),
        },
        "deltas": deltas,
        "runtime": runtime,
        "conclusion": directional_conclusion(
            deltas["precision"], deltas["recall"]
        ),
    }


In [ ]:
result = run_holdout_experiment(
    legalir_source_path=LEGALIR_SOURCE_PATH,
    corpus_path=CORPUS_PATH,
    model_path=MODEL_PATH,
)

assert result["candidate_sets_identical"] == {
    "queries": 1_023,
    "total_queries": 1_023,
}
assert result["candidate_recall_at_100_ceiling"][
    "reference_reranked_identical"
]
assert result["deltas"]["recall_at_100"] == 0

OUTPUT_PATH = Path(
    "/kaggle/working/zero_shot_cross_encoder_reranking_holdout_results.json"
)
OUTPUT_PATH.write_text(
    json.dumps(result, ensure_ascii=False, separators=(",", ":")),
    encoding="utf-8",
)


In [ ]:
summary = {
    "smoke_test": result["smoke_test"],
    "candidate_sets_identical": result["candidate_sets_identical"],
    "candidate_recall_at_100_ceiling": result[
        "candidate_recall_at_100_ceiling"
    ],
    "reference": {
        "precision": result["reference"]["bundled_scorer"]["precision"],
        "recall": result["reference"]["bundled_scorer"]["recall"],
        "mrr": result["reference"]["internal"]["mrr"],
    },
    "reranked": {
        "precision": result["reranked"]["bundled_scorer"]["precision"],
        "recall": result["reranked"]["bundled_scorer"]["recall"],
        "mrr": result["reranked"]["internal"]["mrr"],
    },
    "deltas": result["deltas"],
    "runtime": result["runtime"],
    "conclusion": result["conclusion"],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
